# Long-read DeepVariant **within-population** PCA (Hail / Terra)

Supplemental ancestry PCs within each sufficiently large
`ancestry_pred_other` group. Run **after**:

1. `tractor_05_pca_deepvariant_long_read.ipynb` has written
   `checkpoints/qc_for_pca.mt` under the same `PCA_RUN_LABEL`.
2. `tractor_06_fill_lr_ancestry.ipynb` (or equivalent) has filled
   `ancestry_pred_other` / `population` for joint-callset samples, including
   HG/NA controls.

This notebook reloads the QC MatrixTable, annotates samples from covariates,
and re-prunes / runs PCA per population. It does **not** re-import the joint VCF.

## Covariates

Default path (override with `PCA_COV_URI`):

- Terra: `gs://$WORKSPACE_BUCKET/covariates/covariates.source_rebuilt.csv.gz`
- Local: `tractor_mix/covariates.source_rebuilt.csv.gz`

Upload the filled file to that GCS path (or set `PCA_COV_URI` to wherever it lives)
before running. Subsetting key is `PCA_POPULATION_LABEL` (default
`ancestry_pred_other`). Groups with `n < PCA_MIN_POPULATION_N` (default 100) are
skipped.

## Cluster

Reuse **16–32** `n1-highmem-8` / `n2-highmem-8` workers (no VCF import).
Budget wall time for ~6–7 ancestry subsets (each re-prunes independently).
Use `LD_MEMORY_PER_CORE=1` (integer GiB; Hail does not want `"1g"`).

## Outputs

Under the same `OUT_DIR` as global PCA:

- `population_pcs.tsv` with `lr_pop_PC1`–`lr_pop_PC32`
- per-population loadings / eigenvalues
- `run_metadata.within_population.json`

Re-runs skip work when `population_pcs.tsv` already exists with matching params.
Set `PCA_FORCE_WITHIN_POPULATION=true` to recompute.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

if os.environ.get("WORKSPACE_BUCKET", "").strip():
    os.environ.setdefault("PCA_SYNC_SCRIPTS", "true")

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py",
    "reference_control_ancestry.py",
    "hail_pca_resume.py",
)
from workspace_paths import data_root
from reference_control_ancestry import fill_reference_control_ancestry, is_reference_control
from hail_pca_resume import describe_stage, env_flag, print_stage_plan, write_json_uri

import json

import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = data_root()
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
RUN_LABEL = os.environ.get("PCA_RUN_LABEL", "deepvariant_lr_v2")

if WORKSPACE_BUCKET:
    _bucket = (
        WORKSPACE_BUCKET
        if WORKSPACE_BUCKET.startswith("gs://")
        else f"gs://{WORKSPACE_BUCKET}"
    )
    COV_URI = os.environ.get(
        "PCA_COV_URI", f"{_bucket}/covariates/covariates.source_rebuilt.csv.gz"
    )
    OUT_DIR = os.environ.get("PCA_OUT_DIR", f"{_bucket}/pca/{RUN_LABEL}")
else:
    COV_URI = os.environ.get(
        "PCA_COV_URI", str(ROOT / "covariates.source_rebuilt.csv.gz")
    )
    OUT_DIR = os.environ.get("PCA_OUT_DIR", str(ROOT / "pca" / RUN_LABEL))

N_PCS = int(os.environ.get("PCA_N_PCS", "32"))
MIN_AF = 0.01
MAX_AF = 0.99
MIN_VARIANT_CALL_RATE = 0.98
LD_R2 = 0.1
LD_BP_WINDOW = 500_000
LD_MEMORY_PER_CORE = int(os.environ.get("PCA_LD_MEMORY_PER_CORE", "1"))
MIN_POPULATION_N = int(os.environ.get("PCA_MIN_POPULATION_N", "100"))
POPULATION_LABEL = os.environ.get("PCA_POPULATION_LABEL", "ancestry_pred_other")
RUN_PIPELINE = os.environ.get("PCA_RUN_PIPELINE", "").lower() in {"1", "true", "yes"}

CHECKPOINT_MT = os.environ.get("PCA_CHECKPOINT_MT", f"{OUT_DIR}/checkpoints/qc_for_pca.mt")
POP_PCS_TSV = f"{OUT_DIR}/population_pcs.tsv"
WITHIN_POP_PARAMS_JSON = f"{OUT_DIR}/within_population.params.json"
METADATA_JSON = f"{OUT_DIR}/run_metadata.within_population.json"

FORCE_WITHIN_POP = env_flag("PCA_FORCE_WITHIN_POPULATION")
WITHIN_POP_PARAMS = {
    "population_label": POPULATION_LABEL,
    "min_population_n": MIN_POPULATION_N,
    "cov_uri": COV_URI,
    "checkpoint_mt": CHECKPOINT_MT,
    "n_pcs": N_PCS,
    "ld_r2": LD_R2,
    "ld_bp_window": LD_BP_WINDOW,
    "ld_memory_per_core": LD_MEMORY_PER_CORE,
    "min_af": MIN_AF,
    "max_af": MAX_AF,
    "min_variant_call_rate": MIN_VARIANT_CALL_RATE,
}

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("CHECKPOINT_MT:", CHECKPOINT_MT)
print("COV_URI:", COV_URI)
print("POPULATION_LABEL:", POPULATION_LABEL)
print("MIN_POPULATION_N:", MIN_POPULATION_N)
print("LD_MEMORY_PER_CORE:", LD_MEMORY_PER_CORE)
print("FORCE_WITHIN_POP:", FORCE_WITHIN_POP)
print("RUN_PIPELINE:", RUN_PIPELINE)


In [ ]:
import hail as hl

if RUN_PIPELINE:
    hl.init(default_reference="GRCh38", idempotent=True)
    print("Hail version:", hl.version())
else:
    print("Dry run: Hail is not initialized until RUN_PIPELINE=True")


## 1. Load updated covariates and QC MatrixTable

Point `PCA_COV_URI` at the filled covariates file if it is not at the default
path. Samples without a usable `POPULATION_LABEL` are skipped for within-pop PCA
(they already have global PCs from `tractor_05`).

The QC MatrixTable from global PCA has no sample ancestry annotations; this
section joins labels from covariates onto MT columns.


In [ ]:
def read_covariates(uri: str) -> pd.DataFrame:
    return pd.read_csv(uri, dtype={"research_id": str}, low_memory=False)


def _is_missing_label(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    return series.isna() | s.eq("") | s.str.upper().eq("NA") | s.str.lower().eq("nan")


_ANN_COLS = [
    "research_id",
    "final_releasable_v9",
    "ancestry_pred",
    "ancestry_pred_other",
    "population",
]


def build_sample_ann(mt_samples: list[str], cov_df: pd.DataFrame) -> pd.DataFrame:
    vcf_df = pd.DataFrame({"s": sorted({str(s) for s in mt_samples})})
    cov_ann = cov_df[[c for c in _ANN_COLS if c in cov_df.columns]].rename(
        columns={"research_id": "s"}
    )
    sample_ann = vcf_df.merge(cov_ann, on="s", how="left", validate="one_to_one")
    sample_ann["is_reference_control"] = sample_ann["s"].map(is_reference_control)
    if "final_releasable_v9" in sample_ann.columns:
        sample_ann["final_releasable_v9"] = (
            sample_ann["final_releasable_v9"].fillna(False).astype(bool)
        )
    else:
        sample_ann["final_releasable_v9"] = False
    # Safety net if any HG/NA rows still lack ancestry in the covariates file.
    sample_ann = fill_reference_control_ancestry(sample_ann)
    if POPULATION_LABEL in sample_ann.columns:
        # Normalize ancestry-style labels to lowercase; keep "NA" for missing.
        raw = sample_ann[POPULATION_LABEL]
        missing = _is_missing_label(raw)
        sample_ann[POPULATION_LABEL] = raw.map(
            lambda v: "NA" if pd.isna(v) else str(v).strip().lower()
        )
        sample_ann.loc[missing, POPULATION_LABEL] = "NA"
    sample_ann["s"] = sample_ann["s"].astype(str)
    for col in sample_ann.columns:
        if col in {"s", "final_releasable_v9", "is_reference_control", POPULATION_LABEL}:
            continue
        sample_ann[col] = sample_ann[col].map(lambda v: "NA" if pd.isna(v) else str(v))
    return sample_ann


def annotate_samples(mt, sample_df: pd.DataFrame):
    pdf = sample_df.copy()
    pdf["s"] = pdf["s"].astype(str)
    if "final_releasable_v9" in pdf.columns:
        pdf["final_releasable_v9"] = pdf["final_releasable_v9"].fillna(False).astype(bool)
    if "is_reference_control" in pdf.columns:
        pdf["is_reference_control"] = pdf["is_reference_control"].fillna(False).astype(bool)
    for col in pdf.columns:
        if col in {"s", "final_releasable_v9", "is_reference_control"}:
            continue
        pdf[col] = pdf[col].map(lambda v: "NA" if pd.isna(v) else str(v))
    ht = hl.Table.from_pandas(pdf).key_by("s")
    return mt.annotate_cols(**ht[mt.s])


if RUN_PIPELINE:
    cov = read_covariates(COV_URI)
    assert cov["research_id"].is_unique
    if POPULATION_LABEL not in cov.columns:
        raise KeyError(
            f"POPULATION_LABEL={POPULATION_LABEL!r} not in covariates columns. "
            f"Available: {sorted(cov.columns)}"
        )

    mt = hl.read_matrix_table(CHECKPOINT_MT)
    n_var, n_samp = mt.count()
    print(f"Loaded QC MT: {n_var:,} variants x {n_samp:,} samples")

    sample_ann = build_sample_ann(mt.s.collect(), cov)
    labeled = sample_ann.loc[sample_ann[POPULATION_LABEL].ne("NA")]
    counts = (
        labeled[POPULATION_LABEL]
        .value_counts()
        .rename_axis(POPULATION_LABEL)
        .reset_index(name="n")
    )
    counts["will_run_pca"] = counts["n"] >= MIN_POPULATION_N
    display(counts)
    print(f"Samples with {POPULATION_LABEL}: {len(labeled):,} / {len(sample_ann):,}")
    print(
        "Populations meeting MIN_POPULATION_N="
        f"{MIN_POPULATION_N}: {int(counts['will_run_pca'].sum())}"
    )
    n_ctrl = int(sample_ann["is_reference_control"].fillna(False).sum())
    n_ctrl_labeled = int(
        sample_ann.loc[
            sample_ann["is_reference_control"].fillna(False)
            & sample_ann[POPULATION_LABEL].ne("NA"),
            POPULATION_LABEL,
        ].shape[0]
    )
    print(f"Reference controls in MT: {n_ctrl:,} (labeled: {n_ctrl_labeled:,})")

    mt = annotate_samples(mt, sample_ann)
else:
    print("Dry run: covariate / MT load skipped")


## 2. Within-population LD-pruned PCA


In [ ]:
def run_pca(mt, *, n_pcs: int):
    pruned = hl.ld_prune(
        mt.GT,
        r2=LD_R2,
        bp_window_size=LD_BP_WINDOW,
        memory_per_core=LD_MEMORY_PER_CORE,
    )
    mt_pruned = mt.filter_rows(hl.is_defined(pruned[mt.row_key]))
    eigenvalues, scores, loadings = hl.hwe_normalized_pca(
        mt_pruned.GT,
        k=n_pcs,
        compute_loadings=True,
    )
    return scores, loadings, eigenvalues


def scores_to_dataframe(scores, *, prefix: str) -> pd.DataFrame:
    pdf = scores.to_pandas()
    pc_cols = [f"{prefix}{i}" for i in range(1, N_PCS + 1)]
    expanded = pd.DataFrame(pdf["scores"].tolist(), columns=pc_cols)
    out = pd.concat([pdf[["s"]].rename(columns={"s": "research_id"}), expanded], axis=1)
    assert out["research_id"].is_unique
    return out


within_pop_stage = describe_stage(
    "within-population PCA",
    POP_PCS_TSV,
    force=FORCE_WITHIN_POP,
    params_path=WITHIN_POP_PARAMS_JSON,
    current_params=WITHIN_POP_PARAMS,
)
print_stage_plan([within_pop_stage])

if RUN_PIPELINE:
    if within_pop_stage["action"] == "skip":
        print(f"RESUME: using existing {POP_PCS_TSV}")
        population_pcs = pd.read_csv(POP_PCS_TSV, sep="\t", dtype={"research_id": str})
        display(population_pcs.groupby("population").size().rename("n").reset_index())
    else:
        pop_tables = []
        pop_values = mt.aggregate_cols(hl.agg.counter(mt[POPULATION_LABEL]))
        for population, n in sorted(pop_values.items(), key=lambda item: (str(item[0]), -item[1])):
            if population is None or population == "NA" or n < MIN_POPULATION_N:
                print(f"skip {population!r}: n={n}")
                continue
            print(f"PCA for {population}: n={n}")
            mt_pop = mt.filter_cols(mt[POPULATION_LABEL] == population)
            mt_pop = hl.variant_qc(mt_pop)
            mt_pop = mt_pop.filter_rows(
                (mt_pop.variant_qc.AF[1] >= MIN_AF)
                & (mt_pop.variant_qc.AF[1] <= MAX_AF)
                & (mt_pop.variant_qc.call_rate >= MIN_VARIANT_CALL_RATE)
            )
            scores, loadings, eigenvalues = run_pca(mt_pop, n_pcs=N_PCS)
            pdf = scores_to_dataframe(scores, prefix="lr_pop_PC")
            pdf.insert(1, "population", population)
            pop_tables.append(pdf)
            loadings.write(f"{OUT_DIR}/populations/{population}/loadings.ht", overwrite=True)
            with hl.hadoop_open(f"{OUT_DIR}/populations/{population}/eigenvalues.txt", "w") as handle:
                for value in eigenvalues:
                    handle.write(f"{value}\n")
        if not pop_tables:
            raise RuntimeError("No populations passed MIN_POPULATION_N; check ancestry labels")
        population_pcs = pd.concat(pop_tables, ignore_index=True)
        assert population_pcs["research_id"].is_unique
        hl.Table.from_pandas(population_pcs).export(POP_PCS_TSV)
        write_json_uri(WITHIN_POP_PARAMS_JSON, WITHIN_POP_PARAMS)
        display(population_pcs.groupby("population").size().rename("n").reset_index())
        print("wrote", POP_PCS_TSV)
else:
    print("Dry run: within-population PCA skipped")


## 3. Provenance


In [ ]:
from datetime import datetime, timezone

metadata = {
    "run_label": RUN_LABEL,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "engine": "hail",
    "pca_scope": "within_population",
    "paths": {
        "workspace_bucket": WORKSPACE_BUCKET,
        "covariates": COV_URI,
        "checkpoint_mt": CHECKPOINT_MT,
        "out_dir": OUT_DIR,
        "population_pcs": POP_PCS_TSV,
    },
    "cohort": {
        "within_population_label_source": POPULATION_LABEL,
        "min_population_n": MIN_POPULATION_N,
    },
    "filters": {
        "min_af": MIN_AF,
        "max_af": MAX_AF,
        "min_variant_call_rate": MIN_VARIANT_CALL_RATE,
        "ld_r2": LD_R2,
        "ld_bp_window": LD_BP_WINDOW,
        "ld_memory_per_core": LD_MEMORY_PER_CORE,
        "n_pcs": N_PCS,
    },
    "resume": {
        "within_pop_stage": within_pop_stage if RUN_PIPELINE else None,
        "force_within_population": FORCE_WITHIN_POP,
    },
    "status": "completed" if RUN_PIPELINE else "dry_run",
}

if RUN_PIPELINE:
    with hl.hadoop_open(METADATA_JSON, "w") as handle:
        handle.write(json.dumps(metadata, indent=2) + "\n")
else:
    local_out = ROOT / "pca" / RUN_LABEL
    local_out.mkdir(parents=True, exist_ok=True)
    (local_out / "run_metadata.within_population.dry_run.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )
    print("wrote", local_out / "run_metadata.within_population.dry_run.json")

print(json.dumps(metadata, indent=2))
